NLP Assignment 

In [1]:
import requests
import pandas as pd
import time

API_KEY = '8265bd1679663a7ea12ac168da84d2e8'

# Get genre list to map genre_ids → genre names
genre_url = f'https://api.themoviedb.org/3/genre/movie/list?api_key={API_KEY}&language=en-US'
genre_response = requests.get(genre_url)
genre_data = genre_response.json()['genres']
genre_dict = {g['id']: g['name'] for g in genre_data}

# Store movie data
movie_data = []

# Iterate over all 471 pages of top-rated movies
total_pages = 471

for page in range(1, total_pages + 1):
    print(f"Fetching page {page}/{total_pages}...")
    url = f'https://api.themoviedb.org/3/movie/top_rated?api_key={API_KEY}&language=en-US&page={page}'
    response = requests.get(url)
    
    if response.status_code != 200:
        print(f"⚠️ Failed to fetch page {page}. Skipping...")
        continue

    movies = response.json().get('results', [])
    
    for movie in movies:
        name = movie.get('title', '').strip()
        description = movie.get('overview', '').strip()
        genre_ids = movie.get('genre_ids', [])
        genre = genre_dict[genre_ids[0]] if genre_ids else "Unknown"
        movie_data.append([name, description, genre])

    # Respect TMDb rate limits (max 4 requests/sec)
    time.sleep(0.3)

# Create a DataFrame and save as CSV
df = pd.DataFrame(movie_data, columns=['Title', 'Description', 'Genre'])
df.to_csv('/Users/tanmayjaipuriar/Downloads/Mac learn/Prac/top_rated_movies_dataset.csv', index=False)

print("✅ Dataset saved as 'top_rated_movies_dataset.csv' with", len(df), "records.")


/Users/tanmayjaipuriar/Library/Python/3.9/lib/python/site-packages/urllib3/__init__.py:35: NotOpenSSLWarning: urllib3 v2 only supports OpenSSL 1.1.1+, currently the 'ssl' module is compiled with 'LibreSSL 2.8.3'. See: https://github.com/urllib3/urllib3/issues/3020
  warnings.warn(


Fetching page 1/471...
Fetching page 2/471...
Fetching page 3/471...
Fetching page 4/471...
Fetching page 5/471...
Fetching page 6/471...
Fetching page 7/471...
Fetching page 8/471...
Fetching page 9/471...
Fetching page 10/471...
Fetching page 11/471...
Fetching page 12/471...
Fetching page 13/471...
Fetching page 14/471...
Fetching page 15/471...
Fetching page 16/471...
Fetching page 17/471...
Fetching page 18/471...
Fetching page 19/471...
Fetching page 20/471...
Fetching page 21/471...
Fetching page 22/471...
Fetching page 23/471...
Fetching page 24/471...
Fetching page 25/471...
Fetching page 26/471...
Fetching page 27/471...
Fetching page 28/471...
Fetching page 29/471...
Fetching page 30/471...
Fetching page 31/471...
Fetching page 32/471...
Fetching page 33/471...
Fetching page 34/471...
Fetching page 35/471...
Fetching page 36/471...
Fetching page 37/471...
Fetching page 38/471...
Fetching page 39/471...
Fetching page 40/471...
Fetching page 41/471...
Fetching page 42/471...
F

In [28]:
df = pd.read_csv('/Users/tanmayjaipuriar/Downloads/Mac learn/Prac/top_rated_movies_dataset.csv')

In [31]:
df['Title'] = df['Title'].astype(str)
df['Description'] = df['Description'].astype(str)
df['Genre'] = df['Genre'].astype(str)

In [32]:
df.head()

,Title,Description,Genre
0,The Shawshank Redemption,Imprisoned in the 1940s for the double murder ...,Drama
1,The Godfather,"Spanning the years 1945 to 1955, a chronicle o...",Drama
2,The Godfather Part II,In the continuing saga of the Corleone crime f...,Drama
3,Schindler's List,The true story of how businessman Oskar Schind...,Drama
4,12 Angry Men,The defense and the prosecution have rested an...,Drama


In [33]:
df.shape


(9420, 3)

In [34]:

df.describe(include='all')

,Title,Description,Genre
count,9420,9420,9420
unique,9089,9414,19
top,Return,Wilbur the pig is scared of the end of the sea...,Drama
freq,13,2,2182


Text Preprocessing 

In [35]:
df.columns = df.columns.str.lower()

In [48]:
df['title'] = df['title'].str.lower()
df['description'] = df['description'].str.lower()
df['genre'] = df['genre'].str.lower()

In [36]:
import re
def removal_html_tags(text):
    """Remove HTML tags from a string."""
    clean = re.compile('<.*?>')
    return re.sub(clean, '', text)
df['description'] = df['description'].apply(removal_html_tags)

In [37]:
import string
def remove_punctuation(text):
    """Remove punctuation from a string."""
    return text.translate(str.maketrans('', '', string.punctuation))

In [38]:
df['description'][2]

'In the continuing saga of the Corleone crime family, a young Vito Corleone grows up in Sicily and in 1910s New York. In the 1950s, Michael Corleone attempts to expand the family business into Las Vegas, Hollywood and Cuba.'

In [39]:
import spacy
nlp = spacy.load("en_core_web_sm")
def tokenize_text(text):
    """Tokenize text using spaCy."""
    if not isinstance(text, str):
        return []
    doc = nlp(text)
    return [token.text for token in doc if not token.is_stop and not token.is_punct]


In [40]:
df['title_tokens'] = df['title'].apply(tokenize_text)
df['description_tokens'] = df['description'].apply(tokenize_text)

In [42]:
df['title_tokens'].head()

0    [Shawshank, Redemption]
1                [Godfather]
2            [Godfather, II]
3          [Schindler, List]
4           [12, Angry, Men]
Name: title_tokens, dtype: object

In [44]:
df['description_tokens'].head()

0    [Imprisoned, 1940s, double, murder, wife, love...
1    [Spanning, years, 1945, 1955, chronicle, ficti...
2    [continuing, saga, Corleone, crime, family, yo...
3    [true, story, businessman, Oskar, Schindler, s...
4    [defense, prosecution, rested, jury, filing, j...
Name: description_tokens, dtype: object

In [45]:
import nltk
nltk.download('wordnet')

[nltk_data] Downloading package wordnet to
[nltk_data]     /Users/tanmayjaipuriar/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!


True

In [46]:
from nltk.stem.wordnet import WordNetLemmatizer

lemmatizer = WordNetLemmatizer()
def lemmatize_tokens(tokens):
    return [lemmatizer.lemmatize(token) for token in tokens]
df['title_lemmas'] = df['title_tokens'].apply(lemmatize_tokens)
df['description_lemmas'] = df['description_tokens'].apply(lemmatize_tokens)

In [47]:
df['title_lemmas'].head()
df['description_lemmas'].head()

0    [Imprisoned, 1940s, double, murder, wife, love...
1    [Spanning, year, 1945, 1955, chronicle, fictio...
2    [continuing, saga, Corleone, crime, family, yo...
3    [true, story, businessman, Oskar, Schindler, s...
4    [defense, prosecution, rested, jury, filing, j...
Name: description_lemmas, dtype: object